<a href="https://colab.research.google.com/github/justin92102512-crypto/-/blob/main/B12092202_%E6%9D%8E%E6%B0%B8%E9%9D%96_0709_Colab_LINE_Bot_with_GEMINI_Rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install Flask pyngrok line-bot-sdk requests --quiet
!pip install google-genai --quiet

In [ ]:
from google.colab import userdata

ngrok_authtoken = userdata.get('NGROK_AUTHTOKEN')
line_channel_access_token = userdata.get('LINE_CHANNEL_ACCESS_TOKEN')
line_channel_secret = userdata.get('LINE_CHANNEL_SECRET')
gemini_api_key = userdata.get('GEMINI_API_KEY')
port = 5051


In [ ]:
import os
from pyngrok import ngrok

In [ ]:
ngrok.kill()

In [ ]:
import requests

ngrok.set_auth_token(ngrok_authtoken)
tunnel = ngrok.connect(5051, name="linebot_tunnel")
webhook_url = tunnel.public_url

print(f"Ngrok URL: {webhook_url}")

# 自動更新 LINE Webhook URL
def update_line_webhook(webhook_url):
    """使用 LINE Messaging API 更新 Webhook URL"""
    url = "https://api.line.me/v2/bot/channel/webhook/endpoint"
    headers = {
        "Authorization": f"Bearer {line_channel_access_token}",
        "Content-Type": "application/json"
    }
    data = {
        "endpoint": webhook_url
    }

    response = requests.put(url, headers=headers, json=data)

    if response.status_code == 200:
        print(f"✅ LINE Webhook URL 已自動更新為：{webhook_url}")
        return True
    else:
        print(f"❌ 更新失敗：{response.status_code} - {response.text}")
        return False

# 執行更新
update_line_webhook(webhook_url)

Ngrok URL: https://unminding-fibrotic-paulita.ngrok-free.dev
✅ LINE Webhook URL 已自動更新為：https://unminding-fibrotic-paulita.ngrok-free.dev


True

In [ ]:
from google import genai
from google.genai.types import Tool, GenerateContentConfig, GoogleSearch

# === 初始化 Google Gemini ===
client = genai.Client(api_key=gemini_api_key)

google_search_tool = Tool(
   google_search=GoogleSearch()
)

chat = client.chats.create(
    model="gemini-2.5-flash",
    config=GenerateContentConfig(
        system_instruction="你是一個中文的AI助手，請用繁體中文回答",
        tools=[google_search_tool],
        response_modalities=["TEXT"],
    )
)

In [ ]:
def stateful_query(payload):
    response = chat.send_message(message=payload)
    return response.text

In [ ]:
result = stateful_query("簡介明新科技大學")
print(result)

明新科技大學（Minghsin University of Science and Technology），簡稱明新科大，是一所位於臺灣新竹縣新豐鄉的私立科技大學。 學校名稱「明新」取自《大學》「在明明德，在新民，在止於至善」的精義，旨在闡揚人類與生俱來的德性與情操，期盼學子能涵養高尚品德，擁有專業學問與優良技術，達到全人發展的境界。 其校訓為「堅毅、求新、創造」。

**歷史沿革**
明新科技大學的前身為明新工業專科學校，創立於1966年3月1日。在黨國元老王宗山先生及多位熱心興學人士的努力下成立，初期設有機械、土木、工業管理三科。1997年7月，奉教育部核准改制為明新技術學院，並於2002年8月（或9月）升格為明新科技大學。2018年12月，學校更名為「明新學校財團法人明新科技大學」。

**辦學願景與目標**
明新科大的願景是「深耕在地、放眼國際」，教育目標為「培養具實務經驗與人文素養之專業人才」。學校致力於成為「一流產業大學」，並達成培育「跨域整合、務實創新、全人學習」專業人才的教育目標。

**學術單位**
明新科技大學現設有六個學院，包括：半導體學院、工程學院、管理學院、民生學院、人文與設計學院、共同教育學院。學校涵蓋20個學系、2個學位學程（含1個博士學位學程）及11個碩士班。

**學校特色**
明新科技大學以其產學合作導向和在特定產業領域的卓越表現而聞名。學校位於新竹縣新豐鄉，鄰近新竹科學園區與新竹工業區，享有豐富的產業資源，並以「產業大學」為辦學定位。

*   **半導體產業人才培育**: 明新科大在半導體領域表現突出，被譽為企業最愛聘用的畢業生來源之一。根據1111人力銀行統計，明新科大在半導體產業界最愛聘用畢業生中名列第四，是唯一入榜的私立科大，與臺灣頂尖大學並列前茅。學校設有半導體學院，並於2022年10月獲教育部核准通過「半導體科技博士學位學程」，為該校首個博士班。
*   **企業最愛與起薪優勢**: 在2022年《遠見》雜誌的「企業最愛公私立技職科大調查」中，明新科大在四大領域獲得兩項企業最愛，起薪排名私校第一，甚至優於部分國立科大。
*   **MUST育才特色**: 學校鎖定半導體、AI、元宇宙、風電綠能等前瞻產業，發展MUST四大育才特色，包括多元學習（Multidisciplinary Learning）、全球視野（Univ

In [ ]:
result2 = stateful_query("校長是誰？")
print(result2)

None


In [ ]:
from flask import Flask, request, abort
import logging
import os
import time
from google.genai import types

from linebot.v3 import (
    WebhookHandler
)
from linebot.v3.exceptions import (
    InvalidSignatureError
)
from linebot.v3.messaging import (
    Configuration,
    ApiClient,
    MessagingApi,
    MessagingApiBlob,
    ReplyMessageRequest,
    TextMessage
)
from linebot.v3.webhooks import (
    MessageEvent,
    TextMessageContent,
    FileMessageContent
)

app = Flask(__name__)

logging.basicConfig(
    level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s"
)
app.logger.setLevel(logging.INFO)

configuration = Configuration(access_token=line_channel_access_token)
handler = WebhookHandler(line_channel_secret)

# 儲存檔案的目錄
UPLOAD_DIR = "/content/uploaded_files"
os.makedirs(UPLOAD_DIR, exist_ok=True)

# 儲存每個使用者的對話 session 和上傳的檔案
user_sessions = {}  # {user_id: {"chat": chat_object, "uploaded_file": gemini_file}}

def get_user_session(user_id):
    """取得或建立使用者的對話 session"""
    if user_id not in user_sessions:
        # 建立新的對話 session
        new_chat = client.chats.create(
            model="gemini-2.5-flash",
            config=GenerateContentConfig(
                system_instruction="你是一個中文的AI助手，請用繁體中文回答。如果使用者有提供參考文件，請根據文件內容回答問題。",
                tools=[google_search_tool],
                response_modalities=["TEXT"],
            )
        )
        user_sessions[user_id] = {
            "chat": new_chat,
            "uploaded_file": None
        }
    return user_sessions[user_id]

def download_line_file(message_id, file_name):
    """從 LINE 下載使用者上傳的檔案"""
    with ApiClient(configuration) as api_client:
        line_bot_blob_api = MessagingApiBlob(api_client)
        file_content = line_bot_blob_api.get_message_content(message_id)

        file_path = os.path.join(UPLOAD_DIR, file_name)

        with open(file_path, 'wb') as f:
            f.write(file_content)

        return file_path

def upload_file_to_gemini(file_path):
    """上傳檔案到 Gemini Files API"""
    uploaded_file = client.files.upload(
        file=file_path,
        config={'display_name': os.path.basename(file_path)}
    )

    # 等待檔案處理完成
    while uploaded_file.state.name == "PROCESSING":
        print("檔案處理中...")
        time.sleep(1)
        uploaded_file = client.files.get(name=uploaded_file.name)

    if uploaded_file.state.name == "FAILED":
        raise Exception("檔案上傳處理失敗")

    return uploaded_file

def query_with_rag(user_id, question):
    """使用 RAG 模式回答問題"""
    session = get_user_session(user_id)
    uploaded_file = session["uploaded_file"]

    if uploaded_file:
        # 有上傳檔案，使用 RAG 模式
        response = client.models.generate_content(
            model="gemini-2.5-flash",
            contents=[
                types.Content(
                    role="user",
                    parts=[
                        types.Part.from_uri(
                            file_uri=uploaded_file.uri,
                            mime_type=uploaded_file.mime_type
                        ),
                        types.Part.from_text(text=f"請根據上述提供的檔案內容，用繁體中文回答這個問題：{question}")
                    ]
                )
            ]
        )
        return response.text
    else:
        # 沒有上傳檔案，使用一般多輪對話
        response = session["chat"].send_message(message=question)
        return response.text

@app.route("/", methods=['POST'])
def callback():
    signature = request.headers['X-Line-Signature']
    body = request.get_data(as_text=True)
    print("BODY: ", body)
    app.logger.info("Request body: " + body)

    try:
        handler.handle(body, signature)
    except InvalidSignatureError:
        app.logger.info("Invalid signature.")
        abort(400)

    return 'OK'

@handler.add(MessageEvent, message=TextMessageContent)
def handle_message(event):
    """處理文字訊息"""
    text = event.message.text
    user_id = event.source.user_id

    with ApiClient(configuration) as api_client:
        line_bot_api = MessagingApi(api_client)

        if text.startswith('AI '):
            prompt = text[3:]
            try:
                # 使用 RAG 或一般對話
                reply_text = query_with_rag(user_id, prompt)

                # 檢查是否有上傳檔案，加上提示
                session = get_user_session(user_id)
                if session["uploaded_file"]:
                    reply_text = f"📄 [RAG 模式]\n\n{reply_text}"

                line_bot_api.reply_message_with_http_info(
                    ReplyMessageRequest(
                        reply_token=event.reply_token,
                        messages=[TextMessage(text=reply_text)]
                    )
                )
            except Exception as e:
                line_bot_api.reply_message_with_http_info(
                    ReplyMessageRequest(
                        reply_token=event.reply_token,
                        messages=[TextMessage(text=f"❌ 發生錯誤：{str(e)}")]
                    )
                )
        elif text == "清除文件":
            # 清除使用者上傳的檔案
            session = get_user_session(user_id)
            session["uploaded_file"] = None
            line_bot_api.reply_message_with_http_info(
                ReplyMessageRequest(
                    reply_token=event.reply_token,
                    messages=[TextMessage(text="✅ 已清除上傳的文件，恢復一般對話模式。")]
                )
            )
        else:
            line_bot_api.reply_message_with_http_info(
                ReplyMessageRequest(
                    reply_token=event.reply_token,
                    messages=[TextMessage(text="請輸入「AI 問題」來開始對話\n或上傳 TXT/PDF 檔案啟用 RAG 模式")]
                )
            )

@handler.add(MessageEvent, message=FileMessageContent)
def handle_file_message(event):
    """處理使用者上傳的檔案"""
    user_id = event.source.user_id
    file_name = event.message.file_name

    with ApiClient(configuration) as api_client:
        line_bot_api = MessagingApi(api_client)

        # 檢查檔案類型
        if not (file_name.endswith('.txt') or file_name.endswith('.pdf')):
            line_bot_api.reply_message_with_http_info(
                ReplyMessageRequest(
                    reply_token=event.reply_token,
                    messages=[TextMessage(text="⚠️ 目前只支援 TXT 或 PDF 檔案")]
                )
            )
            return

        try:
            # 下載檔案
            file_path = download_line_file(event.message.id, file_name)
            print(f"檔案已下載：{file_path}")

            # 上傳到 Gemini
            uploaded_file = upload_file_to_gemini(file_path)
            print(f"檔案已上傳到 Gemini：{uploaded_file.uri}")

            # 儲存到使用者 session
            session = get_user_session(user_id)
            session["uploaded_file"] = uploaded_file

            line_bot_api.reply_message_with_http_info(
                ReplyMessageRequest(
                    reply_token=event.reply_token,
                    messages=[TextMessage(text=f"✅ 檔案「{file_name}」上傳成功！\n\n現在您可以輸入「AI 問題」來詢問關於這份文件的問題。\n\n輸入「清除文件」可恢復一般對話模式。")]
                )
            )
        except Exception as e:
            line_bot_api.reply_message_with_http_info(
                ReplyMessageRequest(
                    reply_token=event.reply_token,
                    messages=[TextMessage(text=f"❌ 檔案處理失敗：{str(e)}")]
                )
            )

if __name__ == "__main__":
    app.run(port=port)

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5051
INFO:werkzeug:Press CTRL+C to quit
INFO:__main__:Request body: {"destination":"U96394896c06e15def781d1b318fc78e0","events":[]}
INFO:werkzeug:127.0.0.1 - - [07/Jan/2026 12:32:12] "POST / HTTP/1.1" 200 -


BODY:  {"destination":"U96394896c06e15def781d1b318fc78e0","events":[]}


INFO:__main__:Request body: {"destination":"U96394896c06e15def781d1b318fc78e0","events":[{"type":"message","message":{"type":"text","id":"595520441213780394","quoteToken":"rK2JXwnDFNcgKGvf-fjlc2DdJPn4_dnXTTuvJqisWp7G6k4Y9uBTDqQXty7LAv5tSicgViKzeLUCW8pXsOBW7XTPm4tGE9I9n57PtZWssFXMfJ5tTb_Xxlc_AaWGWv0rV2l5_QqKFUX9RLuJIb6hKA","markAsReadToken":"5oA8AIbZWuABTZnTcO0EUubbtV7C9pUGae8xlDXSOxiMtUmR5WT1bEDh61CYsZbJXI-wOO1kGHjmF4gk986uHY1b9sPdM8zkleoSBAxH-GFCROOJj7M27gj5SBiwNxOPCQOr6MTAZnLsbd_LIfzzoi4R_jKkDUUkaSRYMLrDU6u-K5SN0cIwuF36b1wkN3073EnbaMR2OXnYilEapfEVWQ","text":"AI 現在幾點了"},"webhookEventId":"01KEC73DF3HZECDNABPKYZ1A06","deliveryContext":{"isRedelivery":false},"timestamp":1767789147560,"source":{"type":"user","userId":"Ub61cc5701cb6aa22ef67c1ca3c38052f"},"replyToken":"9e187424798147ca8c33cb1e970c0c79","mode":"active"}]}


BODY:  {"destination":"U96394896c06e15def781d1b318fc78e0","events":[{"type":"message","message":{"type":"text","id":"595520441213780394","quoteToken":"rK2JXwnDFNcgKGvf-fjlc2DdJPn4_dnXTTuvJqisWp7G6k4Y9uBTDqQXty7LAv5tSicgViKzeLUCW8pXsOBW7XTPm4tGE9I9n57PtZWssFXMfJ5tTb_Xxlc_AaWGWv0rV2l5_QqKFUX9RLuJIb6hKA","markAsReadToken":"5oA8AIbZWuABTZnTcO0EUubbtV7C9pUGae8xlDXSOxiMtUmR5WT1bEDh61CYsZbJXI-wOO1kGHjmF4gk986uHY1b9sPdM8zkleoSBAxH-GFCROOJj7M27gj5SBiwNxOPCQOr6MTAZnLsbd_LIfzzoi4R_jKkDUUkaSRYMLrDU6u-K5SN0cIwuF36b1wkN3073EnbaMR2OXnYilEapfEVWQ","text":"AI 現在幾點了"},"webhookEventId":"01KEC73DF3HZECDNABPKYZ1A06","deliveryContext":{"isRedelivery":false},"timestamp":1767789147560,"source":{"type":"user","userId":"Ub61cc5701cb6aa22ef67c1ca3c38052f"},"replyToken":"9e187424798147ca8c33cb1e970c0c79","mode":"active"}]}


...

INFO:werkzeug:127.0.0.1 - - [07/Jan/2026 12:34:57] "POST / HTTP/1.1" 200 -
INFO:__main__:Request body: {"destination":"U96394896c06e15def781d1b318fc78e0","events":[{"type":"message","message":{"type":"text","id":"595520736660029922","quoteToken":"gB9d32Fg6IEUYAuTK7lRBqX49xPvaB9pyLMVvldU8oktPeIDxeS6GbWqTJS3mMK7_mprYlgmrFrhvXg_ZEUykNHQx_MxBmX8tXljs4GcqU6is_U7bIZiJ3N401FpcLN1bLJfR3WyJMwl5wu9WlxsIQ","markAsReadToken":"9MZqRA1c4qbqffydkqk5HlSeW88uHFKQZhl0F_6HFh1TPDxXtjVZLFG2hajdGqefYKR2jkWtGZMjeK15mG6pOhm9Oa2528aZw_PgNOehey2mHkNYblBRGFEnLRcX_W8NkDQmYwNsz4u5r3p59X69MiAn9fKdPITHda2528TBqInpVxktEKbrHcHzYsMcn3Rloh2wJB9o14qQDrIW9sIn7A","text":"AI 名字叫甚麼？"},"webhookEventId":"01KEC78SQW79RMWNZ6J4T903RF","deliveryContext":{"isRedelivery":false},"timestamp":1767789323781,"source":{"type":"user","userId":"Ub61cc5701cb6aa22ef67c1ca3c38052f"},"replyToken":"af100e22cc854e9faec06e670624090a","mode":"active"}]}


BODY:  {"destination":"U96394896c06e15def781d1b318fc78e0","events":[{"type":"message","message":{"type":"text","id":"595520736660029922","quoteToken":"gB9d32Fg6IEUYAuTK7lRBqX49xPvaB9pyLMVvldU8oktPeIDxeS6GbWqTJS3mMK7_mprYlgmrFrhvXg_ZEUykNHQx_MxBmX8tXljs4GcqU6is_U7bIZiJ3N401FpcLN1bLJfR3WyJMwl5wu9WlxsIQ","markAsReadToken":"9MZqRA1c4qbqffydkqk5HlSeW88uHFKQZhl0F_6HFh1TPDxXtjVZLFG2hajdGqefYKR2jkWtGZMjeK15mG6pOhm9Oa2528aZw_PgNOehey2mHkNYblBRGFEnLRcX_W8NkDQmYwNsz4u5r3p59X69MiAn9fKdPITHda2528TBqInpVxktEKbrHcHzYsMcn3Rloh2wJB9o14qQDrIW9sIn7A","text":"AI 名字叫甚麼？"},"webhookEventId":"01KEC78SQW79RMWNZ6J4T903RF","deliveryContext":{"isRedelivery":false},"timestamp":1767789323781,"source":{"type":"user","userId":"Ub61cc5701cb6aa22ef67c1ca3c38052f"},"replyToken":"af100e22cc854e9faec06e670624090a","mode":"active"}]}


INFO:werkzeug:127.0.0.1 - - [07/Jan/2026 12:35:26] "POST / HTTP/1.1" 200 -
INFO:__main__:Request body: {"destination":"U96394896c06e15def781d1b318fc78e0","events":[{"type":"message","message":{"type":"text","id":"595520770533228600","quoteToken":"aznLCsExMPZJsqqx7q7Ysw_k8Z0Ko22cT6sN3w6yASS_qB65adHOydMQtIA_pQzsfqViMREXg4ZT6sWrVy9ru4-WWD2AExeIuu1eIkevo38f4TqwhP89kYuQVQtn89bdMzp5O18RxrafNNKotvgpow","markAsReadToken":"ihxaiLMMsHgCdENsGSofO6qaPHLMzExfN-3XFRQen45lhgrTF6NqBB9pAXgut-xnrvVc2wsDiIOULmN6-f7FGocHpL00B5bhQ_MNKH83U-MSo7ShtFMQVCkFqMFD3cbOzYeiHp7r3toNR9NWdIDmxso4u39KxtN73p9DIyUKWJ0--d-HNN_zTLxokQYP5ej7z6HCom1_O8ozrKPGWF5vhQ","text":"AI 檔案中還有其他問題嗎"},"webhookEventId":"01KEC79DNNXVP32D64EB8K9156","deliveryContext":{"isRedelivery":false},"timestamp":1767789343976,"source":{"type":"user","userId":"Ub61cc5701cb6aa22ef67c1ca3c38052f"},"replyToken":"8aa5fdf159f84435a9525ebd6730639e","mode":"active"}]}


BODY:  {"destination":"U96394896c06e15def781d1b318fc78e0","events":[{"type":"message","message":{"type":"text","id":"595520770533228600","quoteToken":"aznLCsExMPZJsqqx7q7Ysw_k8Z0Ko22cT6sN3w6yASS_qB65adHOydMQtIA_pQzsfqViMREXg4ZT6sWrVy9ru4-WWD2AExeIuu1eIkevo38f4TqwhP89kYuQVQtn89bdMzp5O18RxrafNNKotvgpow","markAsReadToken":"ihxaiLMMsHgCdENsGSofO6qaPHLMzExfN-3XFRQen45lhgrTF6NqBB9pAXgut-xnrvVc2wsDiIOULmN6-f7FGocHpL00B5bhQ_MNKH83U-MSo7ShtFMQVCkFqMFD3cbOzYeiHp7r3toNR9NWdIDmxso4u39KxtN73p9DIyUKWJ0--d-HNN_zTLxokQYP5ej7z6HCom1_O8ozrKPGWF5vhQ","text":"AI 檔案中還有其他問題嗎"},"webhookEventId":"01KEC79DNNXVP32D64EB8K9156","deliveryContext":{"isRedelivery":false},"timestamp":1767789343976,"source":{"type":"user","userId":"Ub61cc5701cb6aa22ef67c1ca3c38052f"},"replyToken":"8aa5fdf159f84435a9525ebd6730639e","mode":"active"}]}


INFO:werkzeug:127.0.0.1 - - [07/Jan/2026 12:35:48] "POST / HTTP/1.1" 200 -


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
